In [ ]:
%%bash
sudo apt-get -y install intel-mkl
sudo apt remove python3-numpy -y
sudo pip3 uninstall numpy -y
python3 -m pip install -i https://pypi.anaconda.org/intel/simple numpy

cd ~
git clone https://github.com/coin-or-tools/ThirdParty-Mumps
cd ThirdParty-Mumps
./get.Mumps
./configure
make
make install
sudo make install

In [ ]:
%%bash
cd ~
git clone https://github.com/coin-or/Ipopt.git
cd ~/Ipopt
export IPOPTDIR=`pwd`
mkdir $IPOPTDIR/build
cd $IPOPTDIR/build
$IPOPTDIR/configure
make
sudo make install

In [1]:
import casadi as ca
import matplotlib.pyplot as plt
import numpy as np
import time 
v = ca.SX.sym("v")
psi = ca.SX.sym("psi")
l_f = ca.SX.sym("l_f")
l_r = ca.SX.sym("l_r")
steering_angle = ca.SX.sym("steering_angle")


beta = ca.atan((l_r / (l_f + l_r)) * ca.tan(steering_angle))

x_dot = v * ca.cos(psi + beta)
y_dot = v * ca.sin(psi + beta)
psi_dot = (v / l_r) * ca.sin(beta)

f = ca.Function("f", [v, psi, steering_angle, l_f, l_r], [x_dot, y_dot, psi_dot], ["v", "psi", "steering_angle", "l_f", "l_r"], ["x_dot", "y_dot", "psi_dot"])

T = 10
number_of_steps = 100
DT = T / number_of_steps
opti = ca.Opti()
v = opti.variable(number_of_steps)
psi = opti.variable(number_of_steps)
x = opti.variable(number_of_steps)
y = opti.variable(number_of_steps)
steering_angle = opti.variable(number_of_steps)
l_f = opti.parameter()
l_r = opti.parameter()
x_0 = opti.parameter()
y_0 = opti.parameter()
psi_0 = opti.parameter()
x_ref = opti.parameter()
y_ref = opti.parameter()
psi_ref = opti.parameter()
dt = opti.parameter()

derivative_function = f.map(number_of_steps,[False,False,False,True,True],[False,False,False])
# derivative_function = f.map(number_of_steps,'openmp')
# derivative_function = f.map("","1"number_of_steps,[False,False,False,True,True],[False,False,False])
x_vector = ca.vertcat(x_0,x[:])
y_vector = ca.vertcat(y_0,y[:])
psi_vector = ca.vertcat(psi_0,psi[:])
derivatives = derivative_function(v,psi,steering_angle,l_f,l_r)
next_x = x_vector[:-1] + derivatives[0].reshape((-1,1))*dt
next_y = y_vector[:-1] + derivatives[1].reshape((-1,1))*dt
next_psi = psi_vector[:-1] + derivatives[2].reshape((-1,1))*dt

opti.subject_to(x == next_x)
opti.subject_to(y == next_y)
opti.subject_to(psi == next_psi)
opti.subject_to(x[-1] == x_ref)
opti.subject_to(y[-1] == y_ref)
opti.subject_to(psi[-1] == psi_ref)
opti.subject_to(ca.fabs(steering_angle) <= ca.pi/4)
opti.subject_to(ca.fabs(v) <= 20)
opti.minimize(ca.sumsqr(v) + ca.sumsqr(steering_angle)*0.01)
# opti.solver("ipopt",{},{"print_level":0,'linear_solver': 'ma57', })
opti.solver("ipopt",{ "expand":True,"jit":True,'jit_options':{'verbose':False,'flags': ['-Ofast','-march=native' ]}},{"print_level":0,'linear_solver': 'mumps',})

opti.set_value(l_f, 0.13)
opti.set_value(l_r, 0.13)
opti.set_value(x_0, 0)
opti.set_value(y_0, 0)
opti.set_value(psi_0, 0)
opti.set_value(x_ref, 1)
opti.set_value(y_ref, 1)
opti.set_value(psi_ref, np.pi)
opti.set_value(dt, DT)


result = opti.solve()
success = result.stats()["success"]
x_solve = result.value(x_vector)
y_solve = result.value(y_vector)
psi_solve = result.value(psi_vector)

steering_solve = result.value(steering_angle)
v_solve = result.value(v)
#       solver  :   t_proc      (avg)   t_wall      (avg)    n_eval
#        nlp_f  | 314.00us (  1.36us) 300.32us (  1.30us)       231
#        nlp_g  |   2.19ms (  9.48us)   2.09ms (  9.05us)       231
#   nlp_grad_f  | 146.00us (  2.56us) 135.50us (  2.38us)        57
#   nlp_hess_l  | 819.00us ( 14.62us) 811.44us ( 14.49us)        56
#    nlp_jac_g  | 771.00us ( 12.85us) 778.03us ( 12.97us)        60
#        total  |  77.80ms ( 77.80ms)  77.42ms ( 77.42ms)         1


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

      solver  :   t_proc      (avg)   t_wall      (avg)    n_eval
       nlp_f  | 523.00us (  2.26us) 503.32us (  2.18us)       231
       nlp_g  |   2.65ms ( 11.48us)   2.55ms ( 11.02us)       231
  nlp_grad_f  | 217.00us (  3.81us) 200.81us (  3.52us)        57
  nlp_hess_l  | 913.00us ( 16.30us) 903.94us ( 16.14us)        56
   nlp_jac_g  |   1.31ms ( 21.88us)   1.02ms ( 16.92us)        60
       total  |  85.86ms ( 85.86ms)  88.33ms ( 88.33ms)         1


In [28]:
result = opti.solve()

      solver  :   t_proc      (avg)   t_wall      (avg)    n_eval
       nlp_f  |   7.55ms (  1.82us)   6.52ms (  1.57us)      4158
       nlp_g  |  43.07ms ( 10.36us)  40.41ms (  9.72us)      4158
    nlp_grad  | 362.00us ( 21.29us) 359.11us ( 21.12us)        17
  nlp_grad_f  |   3.06ms (  2.98us)   2.69ms (  2.62us)      1026
  nlp_hess_l  |  16.65ms ( 16.51us)  16.40ms ( 16.27us)      1008
   nlp_jac_g  |  15.86ms ( 14.68us)  15.94ms ( 14.76us)      1080
       total  |  85.67ms ( 85.67ms)  84.99ms ( 84.99ms)         1


In [1]:
def time_function(func,r = None, s = 1):
    import time
    import numpy as np
    [buf,f_eval] = func.buffer()
    res = []
    for i in range(func.n_out()):
        res.append(np.zeros(func.sparsity_out(i).shape))
        buf.set_res(i, memoryview(res[-1]))
    args = []
    for i in range(func.n_in()):
        args.append(np.random.randn(*func.sparsity_in(i).shape))
        buf.set_arg(i, memoryview(args[-1]))
    times = []
    tt = time.time()
    if not r:
        t0 = time.time()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        t1 = time.time()
        r = int(np.ceil(s/(t1-t0),))
    for it in range(r):
        t0 = time.time()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        f_eval()
        t1 = time.time()
        times.append((t1-t0)/10 )
    times = np.array(times)
    print(time.time()-tt)
    d = 1
    unit = 's'
    mean = times.mean()
    # mean = 0.0001
    if mean < 1:
        d = 1e-3
        unit = 'ms'
    if mean < 1e-3:
        d = 1e-6
        unit = 'us'
    if mean < 1e-6:
        d = 1e-9
        unit = 'ns'
    print(f'Number of repetitions: {r*10}')
    print(f'Mean: {mean/d: .3f} {unit}; standard deviation: {times.std()/d: .3f} {unit}; min: {times.min()/d: .3f} {unit}; max: {times.max()/d: .3f} {unit}')
    
# j = derivative_function.jacobian().jacobian().wrap_as_needed({"jit":True,"jit_options":{"flags":["-Ofast","-march=native"]}})
# j = derivative_function.expand()

# time_function(j,s=3)
import casadi as ca
import numpy as np
num_samples = 1000
num_positions = 21
map_size = 2
num_obstacle_categories = 1

S_DM = ca.DM(np.random.randn(num_samples,num_positions))
A_DM = ca.DM(np.random.randn(num_samples,num_obstacle_categories))
modul = ca.SX
A_MX = modul.sym('A',num_samples,num_obstacle_categories)
S_MX = modul.sym('S',num_samples,num_positions)

q_MX = ca.SX.sym('q',num_positions)
use_fk = False
if use_fk:
    fk_function = ca.Function('fk',[q_MX],[forward_kinematics(q_MX).T.reshape((-1,1))],{'cse':True})
else:
    fk_function = ca.Function('fk',[q_MX],[q_MX[0:7].T.reshape((-1,1))],{'cse':True})

def make_polyharmonic_kernel_function(feature_size,num_samples):
    func_name = 'polyharmonic_kernel'
    modul = ca.MX
    sample_in = modul.sym('x',feature_size)
    support_vectors = modul.sym('support_vectors',feature_size,num_samples,)
    support_vector = modul.sym('support_vectors',feature_size,1,)
    sample_in_rep = ca.repmat(sample_in,1,num_samples)
    result = ca.norm_2((sample_in-support_vector))
    f = ca.Function(func_name,[sample_in,support_vector],[result],{'is_diff_in':[True,False]})
    return f

polyharmonic_kernel_function = make_polyharmonic_kernel_function(fk_function.numel_out(),num_samples)

fk_S_MX = modul.sym('S',fk_function.numel_out(),num_samples)
q_sample = modul.sym('q',num_positions)
fk_q_sample_sym = modul.sym('q',fk_function.numel_out())
score_sym = polyharmonic_kernel_function(fk_q_sample_sym,fk_S_MX)@(A_MX)

score_function_sym = ca.Function('score',[fk_q_sample_sym,fk_S_MX,A_MX],[ca.cse(score_sym),-ca.inf,0],['q_sample','fk_S','A'],['g','lbg','ubg'],{'is_diff_in':[True,False,False],'jit':False,'cse':True,}) 



In [5]:
import casadi as ca
import numpy as np
num_samples = 1000
num_positions = 21
map_size = 2
num_obstacle_categories = 1

S_DM = ca.DM(np.random.randn(num_samples,num_positions))
A_DM = ca.DM(np.random.randn(num_samples,num_obstacle_categories))
modul = ca.SX
A_MX = modul.sym('A',num_samples,num_obstacle_categories)
S_MX = modul.sym('S',num_samples,num_positions)

q_MX = ca.SX.sym('q',num_positions)
use_fk = False
if use_fk:
    fk_function = ca.Function('fk',[q_MX],[forward_kinematics(q_MX).T.reshape((-1,1))],)
else:
    fk_function = ca.Function('fk',[q_MX],[q_MX[0:7].T.reshape((-1,1))],)

def make_polyharmonic_kernel_function(feature_size,num_samples):
    func_name = 'polyharmonic_kernel'
    modul = ca.MX
    sample_in = modul.sym('x',feature_size)
    support_vectors = modul.sym('support_vectors',feature_size,num_samples,)
    support_vector = modul.sym('support_vectors',feature_size,1,)
    sample_in_rep = ca.repmat(sample_in,1,num_samples)
    result = ca.norm_2((sample_in-support_vector))
    f = ca.Function(func_name,[sample_in,support_vector],[result],{'is_diff_in':[True,False]})
    return f

polyharmonic_kernel_function = make_polyharmonic_kernel_function(fk_function.numel_out(),num_samples)

fk_S_MX = modul.sym('S',fk_function.numel_out(),num_samples)
q_sample = modul.sym('q',num_positions)
fk_q_sample_sym = modul.sym('q',fk_function.numel_out())
score_sym = polyharmonic_kernel_function(fk_q_sample_sym,fk_S_MX)@(A_MX)

score_function_sym = ca.Function('score',[fk_q_sample_sym,fk_S_MX,A_MX],[(score_sym),-ca.inf,0],['q_sample','fk_S','A'],['g','lbg','ubg'],{'is_diff_in':[True,False,False],'jit':False,}) 

time_function(score_function_sym.jacobian().jacobian())
time_function(score_function_sym.jacobian().jacobian().map(6,'openmp'))
time_function(score_function_sym.jacobian().jacobian().map(12,'openmp'))
time_function(score_function_sym.jacobian().jacobian().map(24,'openmp'))

2.066542148590088
Number of repetitions: 1880
Mean:  1.096 ms; standard deviation:  0.206 ms; min:  0.845 ms; max:  2.209 ms
0.9664719104766846
Number of repetitions: 90
Mean:  9.497 ms; standard deviation:  6.659 ms; min:  3.933 ms; max:  25.861 ms
2.7483808994293213
Number of repetitions: 160
Mean:  16.774 ms; standard deviation:  3.776 ms; min:  11.051 ms; max:  23.879 ms
3.135881185531616
Number of repetitions: 170
Mean:  18.099 ms; standard deviation:  6.673 ms; min:  8.127 ms; max:  30.384 ms


In [21]:
time_function(score_function_sym.jacobian().jacobian().wrap_as_needed({"never_inline":False}))
time_function(score_function_sym.jacobian().jacobian().map(2))
time_function(score_function_sym.jacobian().jacobian().map(6,'openmp'))
time_function(score_function_sym.jacobian().jacobian().map(12,'openmp'))
time_function(score_function_sym.jacobian().jacobian().map(24,'openmp'))

1.9270691871643066
Number of repetitions: 1790
Mean:  1.073 ms; standard deviation:  0.428 ms; min:  0.873 ms; max:  6.026 ms
2.060513973236084
Number of repetitions: 1110
Mean:  1.848 ms; standard deviation:  0.123 ms; min:  1.689 ms; max:  2.449 ms


KeyboardInterrupt: 

In [23]:
score_function_sym.jacobian().jacobian().map(6,'openmp').generate()

'ompmap6_jac_jac_score.c'

In [19]:
%%bash
export OMP_NUM_THREADS=1
echo $OMP_NUM_THREADS

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)


1


In [16]:
score_function_sym.jacobian().jacobian().wrap_as_needed({"never_inline":False})

Function(wrap_jac_jac_score:(q_sample[7],fk_S[7x1000],A[1000],out_g[1x1,0nz],out_lbg[1x1,0nz],out_ubg[1x1,0nz],out_jac_g_q_sample[1x7,0nz],out_jac_g_fk_S[1x7000,0nz],out_jac_g_A[1x1000,0nz],out_jac_lbg_q_sample[1x7,0nz],out_jac_lbg_fk_S[1x7000,0nz],out_jac_lbg_A[1x1000,0nz],out_jac_ubg_q_sample[1x7,0nz],out_jac_ubg_fk_S[1x7000,0nz],out_jac_ubg_A[1x1000,0nz])->(jac_jac_g_q_sample_q_sample[7x7],jac_jac_g_q_sample_fk_S[7x7000],jac_jac_g_q_sample_A[7x1000],jac_jac_g_q_sample_out_g[7x1,0nz],jac_jac_g_q_sample_out_lbg[7x1,0nz],jac_jac_g_q_sample_out_ubg[7x1,0nz],jac_jac_g_fk_S_q_sample[7000x7,0nz],jac_jac_g_fk_S_fk_S[7000x7000,0nz],jac_jac_g_fk_S_A[7000x1000,0nz],jac_jac_g_fk_S_out_g[7000x1,0nz],jac_jac_g_fk_S_out_lbg[7000x1,0nz],jac_jac_g_fk_S_out_ubg[7000x1,0nz],jac_jac_g_A_q_sample[1000x7,0nz],jac_jac_g_A_fk_S[1000x7000,0nz],jac_jac_g_A_A[1000x1000,0nz],jac_jac_g_A_out_g[1000x1,0nz],jac_jac_g_A_out_lbg[1000x1,0nz],jac_jac_g_A_out_ubg[1000x1,0nz],jac_jac_lbg_q_sample_q_sample[7x7,0nz],jac_

In [4]:
# from utils.misc import time_function
j = derivative_function.jacobian().jacobian().wrap_as_needed({"jit":True,"jit_options": {"flags":["-Ofast",
                                                                                                  "-march=native",
                                                                                                  
                                                                                                #   " -mtune=intel" 
                                                                                                  ]}})
time_function(j,s=3)

NameError: name 'derivative_function' is not defined

In [13]:
import timeit

code_snippet = """
sum = 0
for i in range(1000):
    sum += i
"""

# Measure execution time
execution_time = timeit.timeit(code_snippet, number=1000)
print(f"Execution time: {execution_time} seconds")


Execution time: 0.04214743000011367 seconds


In [1]:
import sys
print(sys.executable)
sys.path
import os
print(os.environ['LD_PRELOAD'])
# import numpy as np
# x = np.random.randn(27,297)
# y = np.random.randn(27,27)

/usr/bin/python3.11
/usr/lib/x86_64-linux-gnu/libmkl_core.so:/usr/lib/x86_64-linux-gnu/libmkl_intel_thread.so:/usr/lib/x86_64-linux-gnu/libmkl_intel_lp64.so:/usr/lib/x86_64-linux-gnu/libiomp5.so:


In [3]:
import numpy as np
x = np.random.randn(27,297)
y = np.random.randn(27,27)

import time
def bench():
    t0 = time.time()
    for i in range(10000):
        v = x.T@y@x

    print(time.time()-t0)
bench()


0.8434545993804932


In [4]:
import numpy as np
x = np.random.randn(27,297)
y = np.random.randn(27,27)

import time
t0 = time.time()
x = []
for i in range(1000000):
    x.append(3)
print(time.time()-t0)

0.08176016807556152


In [ ]:
%%bash
# export Ipopt_DIR=/usr/local
# export IPOPT_DIR=/usr/local
# export HSL=/usr/local
# export MUMPS_INCLUDE_DIR=/usr/local/include/coin-or/mumps/
# export MUMPS_LIBS_LIST=/usr/local/lib/
# export MUMPS_LIBRARIES=/usr/local/lib/
# export MUMPS=/usr/local/lib/

# export PATH=${PATH}:${IPOPT_DIR}/lib

# export PKG_CONFIG_PATH=${PKG_CONFIG_PATH}:${IPOPT_DIR}/lib/pkgconfig
cd /home/user/ && wget https://github.com/casadi/casadi/releases/download/3.6.5/casadi-3.6.5.tar.gz
export CASADI_SETUP_CMAKE_ARGS="-DWITH_THREAD=ON;-DWITH_BUILD_EIGEN3=ON;-DWITH_OPENMP=ON;-DWITH_IPOPT=ON;-DWITH_HSL=ON"

# cd /home/user/ && echo $Ipopt_DIR
cd /home/user/ && pip install casadi-3.6.5.tar.gz